# Notebook 4 -- Validation on 2020-2021 Drops

**Purpose:** score `models/model_final_xgboost.pkl` (the final 30-feature model from Notebook 3) on the independent 2020-2021 drops. These drops come from different years and were labeled by students rather than the professor.

**Input:** the four 2020-2021 feature files in `data/features/val_2021/`, extracted with the same functions as Notebook 2 (`training_features_{dob,split,dist,expanded}_2021.csv`).


In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW, INTERIM, FEATS, MODELS, OUT = (ROOT / p for p in
    ["data/raw", "data/interim", "data/features", "models", "outputs"])
VAL = FEATS / "val_2021"

In [3]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score

## 1. Load and merge the 2020-2021 features

In [4]:
KEYS = ["event_id", "buff_id", "drop_id"]

df = pd.read_csv(VAL / "training_features_dob_2021.csv")
for name in ["split", "dist", "expanded"]:
    df = df.merge(pd.read_csv(VAL / f"training_features_{name}_2021.csv"), on=KEYS, how="left")

# keep only rows with a valid label
df = df[df["pt"].isin([0, 1])]
print(f"Rows with a valid pt label: {len(df)}")
print(f"Fires: {df['event_id'].nunique()}")
print(df["pt"].value_counts())

Rows with a valid pt label: 24584
Fires: 344
pt
0    12386
1    12198
Name: count, dtype: int64


## 2. Preprocessing (identical to Notebook 3)

Clean pairs only (exactly two rows per `drop_id`), `mean_dob`/`median_dob`/`split_mean_dob` filled with 0, then the paired-difference features.

In [5]:
group_sizes = df.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df[df["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows ({df_model['drop_id'].nunique()} drops)")

for col in ["mean_dob", "median_dob", "split_mean_dob"]:
    df_model[col] = df_model[col].fillna(0)

# paired differences: feature minus the mean of the drop's two sides
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    df_model[col.replace("_pixels", "") + "_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(lambda x: x - x.mean())

for col in ["exp_burned_pixels", "exp_unburned_pixels", "exp_early_pixels"]:
    df_model[col.replace("_pixels", "") + "_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

Clean pairs: 24296 rows (12148 drops)


## 3. Predict with the final XGBoost model

In [6]:
model = joblib.load(MODELS / "model_final_xgboost.pkl")
features = joblib.load(MODELS / "model_features_xgboost.pkl")

missing = [f for f in features if f not in df_model.columns]
assert not missing, f"Missing columns: {missing}"

df_val = df_model.dropna(subset=features + ["pt"]).copy()
print(f"Rows scored: {len(df_val)} ({df_val['drop_id'].nunique()} drops, {df_val['event_id'].nunique()} fires)")

df_val["predicted"] = model.predict(df_val[features])
acc = accuracy_score(df_val["pt"], df_val["predicted"])
print(f"\nAccuracy on 2020-2021: {acc:.4f}")
print("Cross-validated accuracy on 2017-2019 (Notebook 3): 0.915 +/- 0.009")

Rows scored: 23854 (12145 drops, 340 fires)

Accuracy on 2020-2021: 0.8959
Cross-validated accuracy on 2017-2019 (Notebook 3): 0.915 +/- 0.009


## 4. Save the misclassified rows

For error analysis. Written to `outputs/` (git-ignored).

In [7]:
wrong = df_val[df_val["pt"] != df_val["predicted"]]
wrong[KEYS + ["pt", "predicted"]].to_csv(OUT / "failed_predictions_val_2021.csv", index=False)
print(f"Saved {len(wrong)} misclassified rows")

Saved 2483 misclassified rows
